# Actividad 2 — Procesamiento de Datos en Infraestructura Cloud (Databricks)
**Autor:** Jhon Jader Benítez Valderrama  
**Grupo:** 10  
**Empresa:** Portosoft S.A.S  
**Curso:** Big Data  
---

##  Objetivo
Migrar y procesar un conjunto de datos obtenido desde Kaggle, desplegándolo en Databricks Community Edition mediante Spark y SQL, evidenciando:

1. Diseño del esquema de datos.  
2. Configuración del entorno Databricks.  
3. Ingesta del dataset y creación de tabla.  
4. Validaciones usando Spark y SQL.  
5. Comparación entre SQL y Spark.



---


##  Diseño del esquema de datos. 
El dataset seleccionado fue:

**LinkedIn Software Engineer Jobs Dataset**  
Fuente: Kaggle  
Autor: Andrés Ionel  
URL: https://www.kaggle.com/asaniczka/software-engineer-job-postings-linkedin

# descripcion del dataset
El dataset contiene información de ofertas laborales, incluyendo:
- Título del empleo
- Empresa
- Ubicación
- Link
- Fecha de primera aparición
- Nivel
- Modalidad
- Habilidades
- Resumen

A continuación prsentamos las variables relevnates

**job_title** *(Título del trabajo)*: Nombre del cargo ofrecido.  
**company** *(Empresa)*: Organización que publica la vacante.  
**job_location** *(Ubicación del trabajo)*: Ciudad/estado/país donde aplica el empleo.  
**job_link** *(Enlace del empleo)*: URL directa a la oferta en LinkedIn.  
**first_seen** *(Fecha de detección)*: Momento en que la oferta fue registrada.  
**search_city** *(Ciudad de búsqueda)*: Ciudad usada como filtro durante la extracción.  
**search_country** *(País de búsqueda)*: País relacionado con la oferta.  
**job_level** *(Nivel del cargo)*: Seniority (Associate, Mid-senior, etc.).  
**job_type** *(Tipo de trabajo)*: Modalidad del empleo (Remote, Onsite, Hybrid).  
**job_summary** *(Resumen del empleo)*: Texto descriptivo del rol.  
**job_skills** *(Habilidades requeridas)*: Lista de habilidades necesarias.

![Diagrama ER](https://raw.githubusercontent.com/jhonbenitez-source/BigData/main/Untitled.png)

%sql
 📐 Diseño del Esquema de Datos

Para esta actividad se definió una única entidad principal llamada **jobs**, que almacena las ofertas laborales obtenidas desde el dataset de Kaggle.

A continuación, se muestra el DDL del esquema propuesto en Spark SQL:

## DDL del Esquema (Modelo ERD)

```sql
Table company {
  company_id int [pk, increment]                 // Identificador único de la empresa
  company_name varchar                           // Nombre de la empresa
  country varchar                                // País donde opera la empresa
}

Table jobs {
  job_id int [pk, increment]                     // Identificador único del empleo
  job_title varchar                              // Título del empleo
  company_id int [ref: > company.company_id]     // Llave foránea → company
  job_location varchar                            // Ubicación del trabajo
  job_link varchar                                // Enlace del anuncio
  first_seen date                                 // Fecha de publicación
  search_city varchar                             // Ciudad usada para la búsqueda
  search_country varchar                          // País usado para la búsqueda
  job_level varchar                               // Nivel del puesto
  job_type varchar                                // Tipo de modalidad (remota, onsite)
  job_summary text                                // Resumen del empleo
  job_skills text                                 // Habilidades requeridas
}


In [0]:
%sql


CREATE TABLE IF NOT EXISTS portosoft_db.jobs (
    job_title STRING,
    company STRING,
    job_location STRING,
    job_link STRING,
    first_seen DATE,
    search_city STRING,
    search_country STRING,
    job_level STRING,
    job_type STRING,
    job_summary STRING,
    job_skills STRING
);


Entidad relacion

![Diagrama ER](https://raw.githubusercontent.com/jhonbenitez-source/BigData/main/Untitled.png)

2. CONFIGURACIÓN DE DATABRICKS
a. Crear y Configurar un Cluster
En la barra lateral izquierda de tu espacio de trabajo de Databricks, haz clic en el icono de "Compute".
Haz clic en "Create Cluster".
Configura:
Cluster Name: EA3_Actividad2
Databricks Runtime Version: e.g., 13.0 LTS (Scala 2.12, Spark 3.4.0)
Python Version: e.g., 3.12.3
Cluster Mode: Standard
Autoscaling: Activado (min 1 nodo – max 4 nodos)
Worker Type: e.g., 4 vCPU / 16 GB RAM
Haz clic en Create Cluster.
 Esto creará un clúster escalable listo para ejecutar Spark y SQL.

 Como no es posible realizarlo en la versión Free de Databricks estos serían los resultados.

Configuración del Clúster
Nombre del clúster: EA3_Actividad2
Databricks Runtime: 13.0 LTS
Python: 3.12.3
Modo: Standard
Núcleos/RAM: 4 vCPU / 16 GB RAM
Autoscaling: 1–4 nodos
b. Version de Python y Spark
Configuración del SparkContext

Si ejecutáramos el siguiente código:

python for item in spark.sparkContext.getConf().getAll(): print(item)

Esto imprimirá: versión de Spark, configuración de Python, directorios, núcleos y RAM asignados.

Como no es posible en la versión free, solo mostraremos la versión de Spark y Python.

Versión de Spark
Se imprime usando spark.version

Versión de python
Se imprime usando sys.version


In [0]:
# mostrar version de sopark
print(spark.version)

4.0.0


In [0]:
# mostramos la version de python importanto la lubreria sys
import sys
print(sys.version)

3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]


%md
##  Estructura de almacenamiento: DBFS (Databricks File System)

Para esta actividad utilizaremos **DBFS (Databricks File System)** como sistema principal de almacenamiento.  
DBFS es un sistema de archivos distribuido que permite guardar datasets, notebooks, modelos y otros recursos dentro del entorno de Databricks.

### 🔹 ¿Por qué usar DBFS?
- Está disponible en **Databricks Community Edition (CE)**.
- Permite subir archivos desde el navegador.
- Se puede acceder desde Python, SQL y Spark.
- Se integra automáticamente con el clúster.


### Estructura de almacenamiento (DBFS y Volumes)

En Databricks Community Edition existen limitaciones respecto al uso del directorio público `/FileStore`, ya que actualmente se encuentra deshabilitado.  
En un entorno estándar de Databricks (Enterprise), la estructura habitual para almacenar datasets es:
/FileStore/datasets/

Sin embargo, debido a las restricciones de Databricks CE, la carga y gestión de archivos debe realizarse utilizando **Volumes**, los cuales sí están habilitados.




## Obtención de datos de Kaggle


In [0]:
!pip install kagglehub[pandas-datasets]>=0.3.8

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Importamos las librerias

In [0]:
import os
import zipfile
import kagglehub
import pandas as pd 

### funciones para utilizar datos estraidos desde keagle

In [0]:
def download_dataset_zip(url = ""):
        print("Descargando dataset desde Kaggle...")
        dataset_path = kagglehub.dataset_download(url)
        print("Ruta al dataset:", dataset_path)
        return dataset_path
    
def extract_zip_files(dataset_path):
        zip_files = [f for f in os.listdir(dataset_path) if f.endswith('.zip')]
        if zip_files:
            zip_file = os.path.join(dataset_path, zip_files[0])
            extract_dir = os.path.join(dataset_path, "extracted")
            os.makedirs(extract_dir, exist_ok=True)
            print(f"Extrayendo {zip_file} en {extract_dir}...")
            with zipfile.ZipFile(zip_file, "r") as z:
                z.extractall(extract_dir)
            return extract_dir
        else:
            # Si no se encuentra un ZIP, se verifica si existen archivos CSV en la ruta
            csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]
            if csv_files:
                print("No se encontró archivo ZIP pero se detectaron archivos CSV; se asume que el dataset ya se encuentra extraído.")
                return dataset_path
            else:
                raise FileNotFoundError("No se encontró ningún archivo .zip ni archivos .csv en la ruta del dataset")

def create_csv(csv_dir, csv_name=None):
    if csv_name:
        file_path = os.path.join(csv_dir, csv_name)
        print(f"Leyendo {file_path}...")
        df = pd.read_csv(file_path, encoding="latin1")
        print("CSV creado correctamente")
        return df
    else:
        csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]
        if not csv_files:
            raise FileNotFoundError("No se encontraron archivos CSV en el directorio extraído")
        for file in csv_files:
            file_path = os.path.join(csv_dir, file)
            print(f"Leyendo {file_path}...")
            df = pd.read_csv(file_path, encoding="latin1")
        print("CSV creado correctamente")
        return df

### descargamos el dataset

In [0]:
# descargamos el dataset
df = pd.DataFrame()
dataset_path = download_dataset_zip("asaniczka/software-engineer-job-postings-linkedin")
# extraemos el zip
extract_dir = extract_zip_files(dataset_path)
# creamos el csv
df = create_csv(extract_dir)

Descargando dataset desde Kaggle...
Ruta al dataset: /home/spark-89d237a7-31df-4df9-b661-41/.cache/kagglehub/datasets/asaniczka/software-engineer-job-postings-linkedin/versions/3
No se encontró archivo ZIP pero se detectaron archivos CSV; se asume que el dataset ya se encuentra extraído.
Leyendo /home/spark-89d237a7-31df-4df9-b661-41/.cache/kagglehub/datasets/asaniczka/software-engineer-job-postings-linkedin/versions/3/postings.csv...
CSV creado correctamente


In [0]:
# miramos que si este descragado el dataset usando head
df.head(3)



,job_title,company,job_location,job_link,first_seen,search_city,search_country,job level,job_type,job_summary,job_skills
0,C# Software Engineer,E Tech Group,"West Chester, OH",https://www.linkedin.com/jobs/view/c%23-softwa...,2023-12-25,Covington,United States,Associate,Remote,"At E Tech Group, joining our team means joinin...","C#, .NET, WPF, ASP.NET MVC, WebAPI, C++, Progr..."
1,Software Implementation Engineer,Kardex,"Cincinnati, OH",https://www.linkedin.com/jobs/view/software-im...,2023-12-25,Covington,United States,Associate,Remote,Kardex Remstar is looking for a\nSoftware Impl...,"Software Implementation, Software Testing, Sof..."
2,"Senior Software Engineer, Back End (Go, AWS, J...",Jobs for Humanity,"Chattanooga, TN",https://www.linkedin.com/jobs/view/senior-soft...,2023-12-25,Chattanooga,United States,Mid senior,Onsite,Company Description\nJobs for Humanity is part...,"Java, Python, SQL, Node.js, Go, Scala, AWS, GC..."


###Creacion del catalogo y el schema 

In [0]:
%sql
-- 1. Crear catálogo principal para Portosoft
CREATE CATALOG IF NOT EXISTS portosoft_catalog;

-- 2. Crear el esquema donde se almacenarán las tablas del dataset
CREATE SCHEMA IF NOT EXISTS portosoft_catalog.jobs_schema;

-- 3. Crear un volume para almacenar archivos no tabulares (CSV, JSON, imágenes, etc.)
CREATE VOLUME IF NOT EXISTS portosoft_catalog.jobs_schema.jobs_volume;


###convertir de oandas a spark


In [0]:
spark_df = spark.createDataFrame(df)

In [0]:
# renombramos columnas con espacios
spark_df = spark_df.withColumnRenamed("job level", "job_level")


###Creación de la Tabla con Spark

In [0]:
spark_df.write.mode("overwrite").saveAsTable("portosoft_catalog.jobs_schema.tbl_jobs_spk")


### hacemos un DESCRIBE TABLE

In [0]:
%sql
DESCRIBE TABLE portosoft_catalog.jobs_schema.tbl_jobs_spk;


col_name,data_type,comment
job_title,string,null
company,string,null
job_location,string,null
job_link,string,null
first_seen,string,null
search_city,string,null
search_country,string,null
job_level,string,null
job_type,string,null
job_summary,string,null


### Descripción de Datos Tabla creada con Spark

In [0]:
# 1. Cargar la tabla creada con Spark
df_jobs = spark.table("portosoft_catalog.jobs_schema.tbl_jobs_spk")

# 2. Seleccionar las columnas numéricas (si existieran)
numeric_cols = [
    f.name for f in df_jobs.schema
    if f.dataType.typeName() in ('double', 'decimal', 'float', 'integer', 'long')
]

# 3. Mostrar estadísticas descriptivas
#    Si no hay columnas numéricas, describe() aplicará a todas las columnas STRING
if len(numeric_cols) > 0:
    df_numeric_stats = df_jobs.select(*numeric_cols)
    display(df_numeric_stats.describe())
else:
    print("⚠️ La tabla no contiene columnas numéricas. Mostrando describe() general:")
    display(df_jobs.describe())


⚠️ La tabla no contiene columnas numéricas. Mostrando describe() general:


summary,job_title,company,job_location,job_link,first_seen,search_city,search_country,job_level,job_type,job_summary,job_skills
count,9380,9380,9380,9380,9380,9380,9380,9380,9380,9376,9367
mean,null,null,null,null,null,null,null,null,null,null,null
stddev,null,null,null,null,null,null,null,null,null,null,null
min,"#738 - Systems Software Engineer - Up to $120,000 - Seeking Military Veterans",#twiceasnice Recruiting,"Abbey Wood, England, United Kingdom",https://au.linkedin.com/jobs/software-engineer-jobs,2023-12-25,Aberdeen,Australia,Associate,Hybrid,"!!Software Developer - Fully Remote - Salary Up To Â£65,000!! Overview I have the pleasure of being partners with a great company who is looking for a bright developer to help them build and maintain new, innovative sports games in a tight-knit environment. Responsibilities Develop and maintain high-quality, scalable, specifically in back-end game functions Extending web-based customer portal. Translating requirements into clean and efficient code whilst communicating effectively within your team. Planning unit tests and validation procedures to assure quality Requirements Primarily coding in TypeScript, JavaScript and Python. Use of common JavaScript libraries, such as Next.js and Prisma. Knowledge of AWS and/or Postgresql would be very helpful. *Please note this role is not providing any sponsorship, Thank you* If you are a highly experienced Software Developer, then do not hesitate on this amazing opportunity and apply now! If you have any enquiries, please email me at Zeqir.repaj@exploreltd.com. !!Software Developer - Fully Remote - Salary Up To Â£65,000!! Show more Show less","* .NET, * .NET Core, * ASP.NET MVC, * C#, * SQL Server, * Azure, * Angular, * Solid, * DRY, * LINQ, * Multithreading, * Microservices, * RESTful, * Web API, * JavaScript, * HTML5, * Agile"
max,ð³ð³ð³x10 Senior Software Engineers- FINTECH Startup - Interviews Slots this week â Series A Funded â Bristol -ð³ð³ð³,é´»æµ·ç²¾å¯å·¥æ¥­è¡ä»½æéå ¬å¸,"Zeeland, MI",https://www.linkedin.com/jobs/view/x-ray-software-engineer-c%2B%2B-linux-bash-at-codeworks-it-careers-3763552390,2023-12-25,Young,United States,Mid senior,Remote,"ð¨ Senior Software Engineer ð¡ Python, C++ or Rust, AWS - Financial Trading ð© City of London ð° Â£150,000 - Â£180,000 + Significant Bonus and Excellent Benefits Senior Software Engineer - Python, C++ or Rust - Financial Trading - High Frequency Trading. Our client is a highly recognised, financial trading specialist with over Â£48 billion under assets with over 1 million customers across the UK & Europe. Due to a brand new project, we are now seeking an experienced financial services, Senior Software Engineer/Platform Engineer for their newest squad based in London. Under the leadership of the Chief Technology Officer, our client is driving a multi-year project, to engineer and modernise the full technology stack, which includes pricing and analytics, risk management, market data and trade capture and reporting. This is a brand-new phase where we are building out an entirely greenfield Cloud infrastructure to support the whole company. This is a chance to join a small tight-knit team where you will have the opportunity to make an huge impact. Responsibilities: Be part of the team responsible for the major build-out of functionality on the new platform. Provide a vision and directly contribute to the overall architecture of the platform. Be the go-to person for end-users. Own parts of the buildout where you will be responsible for planning, and communication with senior stakeholders. Mentoring junior team-members and upskilling the team. Skills & Experience: AWS Cloud development and architecture experience A team player with excellent communication skills Advanced analytical skills (typically evidenced by a degree in maths, physics, computer science, engineering, etc.) Applied programming skills - Python, Rust, C++ or other major languages. If you are an experienc

In [0]:
%sql
SHOW TABLES IN portosoft_catalog.jobs_schema;

database,tableName,isTemporary
jobs_schema,tbl_jobs_spk,false


### INGESTA DE KAGGLE
CREACIÓN DE TABLA CON SQL

In [0]:
%sql
LIST '/Volumes/portosoft_catalog/jobs_schema/jobs_volume';


path,name,size,modification_time


###Carga en Spark
Lectura de la Data desde el Volumen con spark.read.csv sin crear la tabla

In [0]:
# 1. Ruta del archivo CSV dentro del Volume
ruta_csv_volume = '/Volumes/portosoft_catalog/jobs_schema/jobs_volume/postings.csv'

# 2. Leer el archivo CSV desde el Volume sin crear tabla todavía
df_diagnostico = spark.read.csv(
    ruta_csv_volume,
    header=True,
    inferSchema=True
)

# 3. Mostrar las columnas leídas por Spark
print(df_diagnostico.columns)



['job_title', 'company', 'job_location', 'job_link', 'first_seen', 'search_city', 'search_country', 'job level', 'job_type', 'job_summary', 'job_skills']


###Persistencia
Creación de una Vista Temporal

In [0]:
%sql
-- 1. Crear una vista temporal leyendo el archivo CSV desde el Volume
CREATE OR REPLACE TEMPORARY VIEW raw_csv_view 
USING CSV
OPTIONS (
  'path' = '/Volumes/portosoft_catalog/jobs_schema/jobs_volume/postings.csv',
  'header' = 'true',
  'inferSchema' = 'true',
  'timestampFormat' = 'yyyy-MM-dd HH:mm:ss'
);

-- 2. (Diagnóstico) Muestra las columnas reales leídas.
DESCRIBE raw_csv_view;

col_name,data_type,comment
job_title,string,null
company,string,null
job_location,string,null
job_link,string,null
first_seen,string,null
search_city,string,null
search_country,string,null
job level,string,null
job_type,string,null
job_summary,string,null


Creación de la Tabla con SQL

In [0]:
%sql
CREATE TABLE IF NOT EXISTS portosoft_catalog.jobs_schema.linkedin_jobs_tbl
AS
SELECT 
    job_title,
    company,
    job_location,
    job_link,
    first_seen,
    search_city,
    search_country,
    `job level` AS job_level,
    job_type,
    job_summary,
    job_skills
FROM raw_csv_view;


num_affected_rows,num_inserted_rows


##Count

In [0]:
%sql
SELECT COUNT(*) FROM portosoft_catalog.jobs_schema.linkedin_jobs_tbl

COUNT(*)
334288


## DESCRIBE DETAIL

In [0]:
%sql
DESCRIBE DETAIL portosoft_catalog.jobs_schema.linkedin_jobs_tbl

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,bb7e484d-2a1f-47de-9cfd-7da436cc4165,portosoft_catalog.jobs_schema.linkedin_jobs_tbl,null,,2025-11-23T04:38:09.022Z,2025-11-23T04:38:12.000Z,List(),List(),1,10112141,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


##VALIDACIONES: SPARK & SQL
###METADATOS
####Describe Table
Muestra el esquema de la tabla Delta permanente linkedin_jobs_tbl

In [0]:

%sql
DESCRIBE TABLE portosoft_catalog.jobs_schema.linkedin_jobs_tbl

col_name,data_type,comment
job_title,string,null
company,string,null
job_location,string,null
job_link,string,null
first_seen,string,null
search_city,string,null
search_country,string,null
job_level,string,null
job_type,string,null
job_summary,string,null


####Show Create Table
Prueba que la tabla existe y está registrada permanentemente

In [0]:
%sql
SHOW CREATE TABLE portosoft_catalog.jobs_schema.linkedin_jobs_tbl

createtab_stmt
"CREATE TABLE portosoft_catalog.jobs_schema.linkedin_jobs_tbl ( job_title STRING, company STRING, job_location STRING, job_link STRING, first_seen STRING, search_city STRING, search_country STRING, job_level STRING, job_type STRING, job_summary STRING, job_skills STRING) USING delta COLLATION 'UTF8_BINARY' TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')"


####Spark
spark_df.printSchema(), inspecciona y muestra la estructura del DataFrame de Spark.

In [0]:

spark_df.printSchema()

root
 |-- job_title: string (nullable = true)
 |-- company: string (nullable = true)
 |-- job_location: string (nullable = true)
 |-- job_link: string (nullable = true)
 |-- first_seen: string (nullable = true)
 |-- search_city: string (nullable = true)
 |-- search_country: string (nullable = true)
 |-- job_level: string (nullable = true)
 |-- job_type: string (nullable = true)
 |-- job_summary: string (nullable = true)
 |-- job_skills: string (nullable = true)



## DESCRIPCIÓN DE DATOS
 Para describir el dataset, no utilicé df.describe() porque esta función solo genera estadísticas para columnas numéricas y el dataset está compuesto únicamente por columnas de tipo string.

In [0]:
from pyspark.sql import functions as F

# 1. Cargar la tabla
df = spark.table("portosoft_catalog.jobs_schema.tbl_jobs_spk")

# 2. Detectar columnas string
string_cols = [c for c, t in df.dtypes if t == "string"]

# 3. Construir métricas para cada columna
metrics = []
for col in string_cols:
    metrics.append(
        df.select(
            F.lit(col).alias("column"),
            F.sum(F.when(F.col(col).isNull() | (F.col(col) == ""), 1).otherwise(0)).alias("null_count"),
            F.countDistinct(col).alias("unique_values"),
            F.avg(F.length(F.col(col))).alias("avg_length")
        )
    )

# 4. Unir todas las métricas en un solo DataFrame
df_metrics = metrics[0]
for m in metrics[1:]:
    df_metrics = df_metrics.unionByName(m)

# 5. Mostrar resultados
display(df_metrics)


column,null_count,unique_values,avg_length
job_title,0,3870,35.85948827292111
company,0,3373,14.459381663113007
job_location,0,1618,17.983475479744136
job_link,0,9380,98.71673773987207
first_seen,0,1,10.0
search_city,0,709,9.08091684434968
search_country,0,4,12.502665245202559
job_level,0,2,9.8636460554371
job_type,0,3,6.0
job_summary,4,7763,3935.3380972696245


###Descripcion de datos usando sql

In [0]:
%sql
-- 1️⃣ COUNT (valores no nulos)
SELECT
    'count' AS summary,
    COUNT(job_title) AS job_title,
    COUNT(company) AS company,
    COUNT(job_location) AS job_location,
    COUNT(job_link) AS job_link,
    COUNT(first_seen) AS first_seen,
    COUNT(search_city) AS search_city,
    COUNT(search_country) AS search_country,
    COUNT(job_level) AS job_level,
    COUNT(job_type) AS job_type,
    COUNT(job_summary) AS job_summary,
    COUNT(job_skills) AS job_skills
FROM portosoft_catalog.jobs_schema.tbl_jobs_spk

UNION ALL
-- 2️⃣ NULLS
SELECT
    'nulls' AS summary,
    SUM(CASE WHEN job_title IS NULL THEN 1 END),
    SUM(CASE WHEN company IS NULL THEN 1 END),
    SUM(CASE WHEN job_location IS NULL THEN 1 END),
    SUM(CASE WHEN job_link IS NULL THEN 1 END),
    SUM(CASE WHEN first_seen IS NULL THEN 1 END),
    SUM(CASE WHEN search_city IS NULL THEN 1 END),
    SUM(CASE WHEN search_country IS NULL THEN 1 END),
    SUM(CASE WHEN job_level IS NULL THEN 1 END),
    SUM(CASE WHEN job_type IS NULL THEN 1 END),
    SUM(CASE WHEN job_summary IS NULL THEN 1 END),
    SUM(CASE WHEN job_skills IS NULL THEN 1 END)
FROM portosoft_catalog.jobs_schema.tbl_jobs_spk

UNION ALL
-- 3️⃣ EMPTY STRINGS ("")
SELECT
    'empty' AS summary,
    SUM(CASE WHEN job_title = '' THEN 1 END),
    SUM(CASE WHEN company = '' THEN 1 END),
    SUM(CASE WHEN job_location = '' THEN 1 END),
    SUM(CASE WHEN job_link = '' THEN 1 END),
    SUM(CASE WHEN first_seen = '' THEN 1 END),
    SUM(CASE WHEN search_city = '' THEN 1 END),
    SUM(CASE WHEN search_country = '' THEN 1 END),
    SUM(CASE WHEN job_level = '' THEN 1 END),
    SUM(CASE WHEN job_type = '' THEN 1 END),
    SUM(CASE WHEN job_summary = '' THEN 1 END),
    SUM(CASE WHEN job_skills = '' THEN 1 END)
FROM portosoft_catalog.jobs_schema.tbl_jobs_spk

UNION ALL
-- 4️⃣ DISTINCT VALUES
SELECT
    'distinct' AS summary,
    COUNT(DISTINCT job_title),
    COUNT(DISTINCT company),
    COUNT(DISTINCT job_location),
    COUNT(DISTINCT job_link),
    COUNT(DISTINCT first_seen),
    COUNT(DISTINCT search_city),
    COUNT(DISTINCT search_country),
    COUNT(DISTINCT job_level),
    COUNT(DISTINCT job_type),
    COUNT(DISTINCT job_summary),
    COUNT(DISTINCT job_skills)
FROM portosoft_catalog.jobs_schema.tbl_jobs_spk

UNION ALL
-- 5️⃣ AVG LENGTH
SELECT
    'avg_length' AS summary,
    AVG(LENGTH(job_title)),
    AVG(LENGTH(company)),
    AVG(LENGTH(job_location)),
    AVG(LENGTH(job_link)),
    AVG(LENGTH(first_seen)),
    AVG(LENGTH(search_city)),
    AVG(LENGTH(search_country)),
    AVG(LENGTH(job_level)),
    AVG(LENGTH(job_type)),
    AVG(LENGTH(job_summary)),
    AVG(LENGTH(job_skills))
FROM portosoft_catalog.jobs_schema.tbl_jobs_spk;


summary,job_title,company,job_location,job_link,first_seen,search_city,search_country,job_level,job_type,job_summary,job_skills
nulls,null,null,null,null,null,null,null,null,null,4.0,13.0
empty,null,null,null,null,null,null,null,null,null,null,null
distinct,3870.0,3373.0,1618.0,9380.0,1.0,709.0,4.0,2.0,3.0,7763.0,9340.0
avg_length,35.85948827292111,14.459381663113007,17.983475479744136,98.71673773987207,10.0,9.08091684434968,12.502665245202559,9.8636460554371,6.0,3935.3380972696245,265.1407067364151
count,9380.0,9380.0,9380.0,9380.0,9380.0,9380.0,9380.0,9380.0,9380.0,9376.0,9367.0


###CONSULTAS EN SQL SELECT Y GROUP BY
esta consulta nos dice cuantas ofertas por empresa hay en nuestra db

####con sql

In [0]:
%sql
SELECT 
    company,
    COUNT(*) AS total_ofertas
FROM portosoft_catalog.jobs_schema.tbl_jobs_spk
GROUP BY company
ORDER BY total_ofertas DESC
LIMIT 20;

company,total_ofertas
Jobs for Humanity,681
Canonical,289
Recruiting from Scratch,201
Affirm,137
ClearanceJobs,110
IP Recruiter Group,99
Northrop Grumman,97
Get It Recruit - Information Technology,78
Energy Jobline,71
Trimble Inc.,68


#### con spark

In [0]:
from pyspark.sql import functions as F

# Cargar la tabla como DataFrame Spark
df_jobs = spark.table("portosoft_catalog.jobs_schema.tbl_jobs_spk")

# Agrupar, contar y limitar a 20
df_ofertas_empresa = (
    df_jobs
    .groupBy("company")
    .agg(F.count("*").alias("total_ofertas"))
    .orderBy(F.desc("total_ofertas"))
    .limit(20)  # <--- Aquí aplicamos el límite
)

display(df_ofertas_empresa)

company,total_ofertas
Jobs for Humanity,681
Canonical,289
Recruiting from Scratch,201
Affirm,137
ClearanceJobs,110
IP Recruiter Group,99
Northrop Grumman,97
Get It Recruit - Information Technology,78
Energy Jobline,71
Trimble Inc.,68


###conteos y muestras
#### conteo de registros y valores distintos
#####con sql

In [0]:
%sql
-- Total de registros
SELECT COUNT(*) AS total_ofertas
FROM portosoft_catalog.jobs_schema.tbl_jobs_spk;

-- Conteo de valores distintos
SELECT
    COUNT(DISTINCT company) AS empresas_distintas,
    COUNT(DISTINCT job_title) AS titulos_distintos,
    COUNT(DISTINCT job_location) AS ubicaciones_distintas,
    COUNT(DISTINCT job_type) AS tipos_trabajo_distintos
FROM portosoft_catalog.jobs_schema.tbl_jobs_spk;


empresas_distintas,titulos_distintos,ubicaciones_distintas,tipos_trabajo_distintos
3373,3870,1618,3


#####con spark

In [0]:
from pyspark.sql import functions as F

df = spark.table("portosoft_catalog.jobs_schema.tbl_jobs_spk")

# Total de registros
total = df.count()
print("Total de registros:", total)


display(df_nulls)

# Conteo de valores distintos por columna clave
df_counts = df.agg(
    F.countDistinct("company").alias("empresas_distintas"),
    F.countDistinct("job_title").alias("titulos_distintos"),
    F.countDistinct("job_location").alias("ubicaciones_distintas"),
    F.countDistinct("job_type").alias("tipos_trabajo_distintos")
)

display(df_counts)


Total de registros: 9380


job_title_nulls,company_nulls,job_location_nulls,job_link_nulls,first_seen_nulls,search_city_nulls,search_country_nulls,job_level_nulls,job_type_nulls,job_summary_nulls,job_skills_nulls
0,0,0,0,0,0,0,0,0,4,13


empresas_distintas,titulos_distintos,ubicaciones_distintas,tipos_trabajo_distintos
3373,3870,1618,3


### sql vs spark: ventajas y desventajas

![Diferencias netre tecnologias](https://raw.githubusercontent.com/jhonbenitez-source/BigData/main/comparacion.png)
jhonbenitez-source/BigData/main/comparacion.png